In [16]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10780\329546744.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [17]:

def process_all_pdf(pdf_directory):
    all_documents = []

    pdf_directory = Path(pdf_directory)
    pdf_files = list(pdf_directory.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files.")

    for pdf_file in pdf_files:
        try:
            loader = PyMuPDFLoader(str(pdf_file))

            documents = loader.load()

            for doc in documents:
                doc.metadata["source_filename"] = pdf_file.name
                doc.metadata["file_path"] = str(pdf_file)

            # Add the list of Document objects
            all_documents.extend(documents)

            print(f"✓ {pdf_file.name} -> {len(documents)} pages")

        except Exception as e:
            print(f"❌ Error processing {pdf_file.name}: {e}")

    print(f"\nTotal Documents: {len(all_documents)}")

    return all_documents


In [18]:
all_documents = process_all_pdf("data/pdf")

print(type(all_documents))
print(type(all_documents[0]))
print(all_documents[0])

Found 6 PDF files.
✓ AI_RAG.pdf -> 1 pages
✓ Company_Overview.pdf -> 1 pages
✓ Contact.pdf -> 1 pages
✓ FAQ.pdf -> 1 pages
✓ Services.pdf -> 1 pages
✓ Technologies.pdf -> 1 pages

Total Documents: 6
<class 'list'>
<class 'langchain_core.documents.base.Document'>
page_content='AI & RAG
Unistack Solutions develops Retrieval-Augmented Generation (RAG) systems using LangChain,
LangGraph, OpenAI-compatible models, embeddings and vector databases such as ChromaDB,
Qdrant, FAISS and Pinecone.
Typical workflow: Load Documents → Chunk Text → Create Embeddings → Store in Vector DB →
Similarity Search → Generate LLM Response.' metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-25T10:10:43+00:00', 'source': 'data\\pdf\\AI_RAG.pdf', 'file_path': 'data\\pdf\\AI_RAG.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-25T10:10:43+00:0

In [19]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
        length_function=len,
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Original Documents : {len(documents)}")
    print(f"Total Chunks       : {len(split_docs)}")

    for doc in split_docs:
        doc.metadata["chunk_index"] = doc.metadata.get("chunk_index", 0)

    return split_docs

In [20]:
chunks = split_documents(all_documents)
print(chunks)

Original Documents : 6
Total Chunks       : 6
[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-25T10:10:43+00:00', 'source': 'data\\pdf\\AI_RAG.pdf', 'file_path': 'data\\pdf\\AI_RAG.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-25T10:10:43+00:00', 'trapped': '', 'modDate': "D:20260625101043+00'00'", 'creationDate': "D:20260625101043+00'00'", 'page': 0, 'source_filename': 'AI_RAG.pdf', 'chunk_index': 0}, page_content='AI & RAG\nUnistack Solutions develops Retrieval-Augmented Generation (RAG) systems using LangChain,\nLangGraph, OpenAI-compatible models, embeddings and vector databases such as ChromaDB,\nQdrant, FAISS and Pinecone.\nTypical workflow: Load Documents → Chunk Text → Create Embeddings → Store in Vector DB →\nSimilarity Search → Generate LLM Response.'), Document(metadata={'producer': 'Repor

In [21]:
import numpy as np

from sentence_transformers import SentenceTransformer

import chromadb
from chromadb.config import Settings

import uuid

from typing import List, Dict, Any, Tuple

from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class EmbeddingManager:

    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(f"Model '{self.model_name}' loaded successfully.")
            print(
                f"Embedding Dimension: {self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, texts):
        """
        Generate embeddings for multiple texts.
        """

        if self.model is None:
            raise ValueError("Embedding model is not loaded.")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        print(f"Generated embeddings: {embeddings.shape}")

        return embeddings

    def generate_embedding(self, text):
        """
        Generate embedding for a single text.
        """

        if self.model is None:
            raise ValueError("Embedding model is not loaded.")

        embedding = self.model.encode(
            text,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        return embedding

    def similarity(self, embedding1, embedding2):
        """
        Calculate cosine similarity.
        """

        similarity = cosine_similarity(
            [embedding1],
            [embedding2]
        )[0][0]

        return similarity

    def get_dimension(self):
        """
        Return embedding dimension.
        """

        if self.model is None:
            raise ValueError("Embedding model is not loaded.")

        return self.model.get_sentence_embedding_dimension()

In [28]:
embedding_manager = EmbeddingManager("all-MiniLM-L6-v2")
embedding_manager

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3981.32it/s]


Model 'all-MiniLM-L6-v2' loaded successfully.
Embedding Dimension: 384


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_24440\679476262.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Embedding Dimension: {self.model.get_sentence_embedding_dimension()}"


In [29]:
import os
class VectorStore:

    def __init__(
        self,
        collection_name="my_collection",
        persist_directory="data/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Vector store for document embeddings"
                }
            )

            print(
                f"Vector store initialized at '{self.persist_directory}'"
            )

            print(
                f"Collection '{self.collection_name}' contains "
                f"{self.collection.count()} documents."
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents, embeddings):

        if len(documents) != len(embeddings):
            raise ValueError(
                "Documents and embeddings must have the same length."
            )

        print(f"Adding {len(documents)} documents...")

        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):

            doc_id = str(uuid.uuid4())

            ids.append(doc_id)

            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["chunk_length"] = len(doc.page_content)

            metadatas.append(metadata)

            documents_texts.append(doc.page_content)

            embeddings_list.append(emb.tolist())

        try:

            self.collection.add(
                ids=ids,
                documents=documents_texts,
                metadatas=metadatas,
                embeddings=embeddings_list,
            )

            print(
                f"Successfully added {len(documents)} documents."
            )

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise

    def search(self, query_embedding, top_k=5):

        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        return results

    def count(self):
        return self.collection.count()

    def delete_collection(self):
        self.client.delete_collection(self.collection_name)
        print(f"Collection '{self.collection_name}' deleted.")

    def reset(self):
        self.client.reset()
        print("Vector store reset successfully.")

In [30]:
vectorStore = VectorStore()
vectorStore


Vector store initialized at 'data/vector_store'
Collection 'my_collection' contains 6 documents.


In [31]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vectorStore.add_documents(chunks, embeddings)

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.46s/it]


Generated embeddings: (6, 384)
Adding 6 documents...
Successfully added 6 documents.


In [ ]:
class RagRetriever:

    def __init__(self, VectorStore, EmbeddingManager):
        self.vector_store = VectorStore
        self.embedding_manager = EmbeddingManager
        
    def retrieve(self, query: str, top_k: int = 5,score_threshold:float =0.0) -> List[Dict[str, Any]]:
        query_embedding = self.embedding_manager.generate_embedding(query)
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs = []
            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]
                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "document": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score
                        })
                    else:
                        print(f"Document {doc_id} filtered out due to low similarity score: {similarity_score:.4f}")
            else:
                print("No documents found for the given query.")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriever = RagRetriever(vectorStore,embedding_manager)

In [41]:
rag_retriever

In [42]:
rag_retriever.retrieve("What is the contact information for Unistack Solutions?", top_k=5)

Document 1c37b47d-ce01-4a69-a978-f09243301a13 filtered out due to low similarity score: 0.1803
Document c2103e5d-84ff-43a0-923b-8629a87e3302 filtered out due to low similarity score: 0.1803
Document 7894f7c3-cb5c-461a-9fec-300c72aaad0a filtered out due to low similarity score: -0.6898


[{'id': '1fffb9ac-342f-4221-8d31-77b57d628d9d',
  'document': 'Contact\nCompany: Unistack Solutions\nWebsite: https://www.unistacksolutions.com\nEmail: info@unistacksolutions.com\nBusiness Hours: Monday-Friday, 9:00 AM - 6:00 PM IST',
  'metadata': {'modDate': "D:20260625101043+00'00'",
   'keywords': '',
   'chunk_index': 0,
   'total_pages': 1,
   'title': '(anonymous)',
   'moddate': '2026-06-25T10:10:43+00:00',
   'creationdate': '2026-06-25T10:10:43+00:00',
   'source_filename': 'Contact.pdf',
   'trapped': '',
   'file_path': 'data\\pdf\\Contact.pdf',
   'author': '(anonymous)',
   'doc_index': 2,
   'format': 'PDF 1.4',
   'subject': '(unspecified)',
   'page': 0,
   'creator': '(unspecified)',
   'source': 'data\\pdf\\Contact.pdf',
   'producer': 'ReportLab PDF Library - (opensource)',
   'creationDate': "D:20260625101043+00'00'",
   'chunk_length': 165},
  'similarity_score': 0.4992656707763672},
 {'id': '16c9a43a-c1d3-4b12-81ca-a8bb31945b52',
  'document': 'Contact\nCompany: 

In [43]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQAPIKEY")

# llm = ChatGroq(api_key=groq_api_key,model="gemma2-9b-it",temperature=0.1,max_tokens=2048)
llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.3-70b-versatile",
    temperature=0,
)

def rag_simple(query,retriever, llm, top_k=5):
    retrieved_docs = retriever.retrieve(query, top_k=top_k)
    if not retrieved_docs:
        return "No relevant documents found."
    
    context = "\n\n".join([f"Document ID: {doc['id']}\nContent: {doc['document']}" for doc in retrieved_docs])
    
    prompt = f"Use the following context to answer the question:\n\n{context}\n\nQuestion: {query}\nAnswer:"
    
    response = llm.invoke([prompt.format(context=context, question=query)])
    
    return response.content

In [45]:
answer = rag_simple("What is the contact information for Unistack Solutions?", rag_retriever, llm)
print(answer)

Document 1c37b47d-ce01-4a69-a978-f09243301a13 filtered out due to low similarity score: 0.1803
Document c2103e5d-84ff-43a0-923b-8629a87e3302 filtered out due to low similarity score: 0.1803
Document 7894f7c3-cb5c-461a-9fec-300c72aaad0a filtered out due to low similarity score: -0.6898


BadRequestError: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}